### Exporting the recommendations to a real Spotify playlist

Goals : take the songs from notebook 05 and actually push them into a playlist on my Spotify account.

> Notebook 05 saved `data/metamood_real_songs.csv`, and every row already carries an exact Spotify `track_id`, so I don't need any fuzzy name matching, I can send the exact tracks straight to a playlist.

> One thing to sort out first : all the earlier notebooks used `SpotifyClientCredentials`, which is the **app-only** flow. It's great for reading the catalog but it **can't create playlists** on my account. For that I need the **user authorization** flow (`SpotifyOAuth`), so there's a tiny one-time setup to do :
> 1. go to my app in the [Spotify Developer Dashboard](https://developer.spotify.com/dashboard) (the same app whose Client ID / Secret are already in my `.env`)
> 2. open **Settings -> Redirect URIs** and add exactly `http://127.0.0.1:8888/callback`
> 3. save
>
> The very first time I run the auth cell below, a browser tab opens to approve it, and after that the token is cached so I'm not asked again.

In [1]:
import os
import spotipy
import pandas as pd
from spotipy.oauth2 import SpotifyOAuth
from dotenv import load_dotenv

load_dotenv()

# Same Client ID / Secret we already use in notebook 01, read from .env
CLIENT_ID = os.getenv("SPOTIFY_CLIENT_ID")
CLIENT_SECRET = os.getenv("SPOTIFY_CLIENT_SECRET")

# This must match EXACTLY what you registered in the dashboard (step 2 above)
REDIRECT_URI = "http://127.0.0.1:8888/callback"

# playlist-modify-private lets us create and fill a private playlist.
# Use playlist-modify-public instead if you want the playlist to be public.
auth_manager = SpotifyOAuth(
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
    redirect_uri=REDIRECT_URI,
    scope="playlist-modify-private",
    # Separate cache file so we don't clobber the app-only token from notebook 01
    cache_path=".cache-user",
)
sp = spotipy.Spotify(auth_manager=auth_manager, requests_timeout=20)

# A quick call that requires a user token. If this prints your name, auth worked.
me = sp.me()
print(f"Authenticated as: {me['display_name']} ({me['id']})")

Authenticated as: Green (7plmwgvr8ya0arrwn6vabo38j)


In [2]:
# Load the recommendations from notebook 05 and turn track_ids into Spotify URIs
recs = pd.read_csv("../data/metamood_real_songs.csv")

# Drop any rows missing a track_id, just in case, and de-duplicate while keeping order
track_ids = recs["track_id"].dropna().drop_duplicates().tolist()
track_uris = [f"spotify:track:{tid}" for tid in track_ids]

print(f"{len(track_uris)} tracks ready to add")
recs[["artists", "track_name"]].head(10)

50 tracks ready to add


,artists,track_name
0,Sufjan Stevens,Fourth of July
1,Yot Club,YKWIM?
2,Billie Eilish,everything i wanted
3,JVKE,golden hour
4,Tom Rosenthal,Lights Are On
5,Billie Eilish,ocean eyes
6,Liana Flores,rises the moon
7,Lord Huron,The Night We Met
8,Billie Eilish;Khalid,lovely (with Khalid)
9,Joji,Glimpse of Us


In [3]:
from spotipy.exceptions import SpotifyException

PLAYLIST_NAME = "MetaTune Recommendations"
PLAYLIST_DESCRIPTION = "Generated by the MetaTune music recommender (notebook 05)."

try:
    playlist = sp.current_user_playlist_create(
        name=PLAYLIST_NAME, public=False, description=PLAYLIST_DESCRIPTION,
    )
    # the API takes at most 100 tracks per call, so I add them in batches
    for start in range(0, len(track_uris), 100):
        sp.playlist_add_items(playlist["id"], track_uris[start:start + 100])
    print(f"Added {len(track_uris)} tracks to '{PLAYLIST_NAME}'")
    print("Open it here:", playlist["external_urls"]["spotify"])
except SpotifyException as e:
    if e.http_status == 403:
        print("Spotify returned 403 Forbidden.")
        print("My app is in Development Mode, and Spotify blocks playlist *writes* for dev-mode")
        print("apps (it lets me read my account, but not create playlists). No problem,")
        print("I'll just export an M3U file below and import it with a free tool instead.")
    else:
        raise

Added 50 tracks to 'MetaTune Recommendations'
Open it here: https://open.spotify.com/playlist/6KCDuJFicLwwCDuIPWcfOI


### A backup : also save the playlist as a file

> The cell above creates the playlist straight through the API, which is exactly what I want. But Spotify apps in **Development Mode** are flaky about write access (it returned a **403 Forbidden** for me earlier, before my account finished propagating on the app's allowlist), so as a safety net I also save the recommendations as an **M3U** file.
>
> Every row already carries an exact `track_id`, so if the API ever refuses again I can just import this file into Spotify with a free tool like [Soundiiz](https://soundiiz.com) or [TuneMyMusic](https://www.tunemymusic.com) and get the exact same playlist in a few seconds, no API needed.

In [4]:
from pathlib import Path

# write an M3U playlist file (track name + open.spotify.com link per track)
m3u_path = Path("../data/metamood_real_songs.m3u8")
lines = ["#EXTM3U"]
for _, r in recs.iterrows():
    lines.append(f"#EXTINF:-1,{r['artists']} - {r['track_name']}")
    lines.append(f"https://open.spotify.com/track/{r['track_id']}")
m3u_path.write_text("\n".join(lines) + "\n", encoding="utf-8")

print(f"{len(recs)} tracks written to: {m3u_path.resolve()}")
print("Import this file into Spotify via Soundiiz or TuneMyMusic.")

50 tracks written to: /Users/mohamedzirar/Documents/year3/sem2/MetaTune/data/metamood_real_songs.m3u8
Import this file into Spotify via Soundiiz or TuneMyMusic.


> And that's it, the **MetaTune Recommendations** playlist now lives on my Spotify account (under **Your Library**) ! And if the API ever blocks the write again, the `.m3u8` backup gets me there too through Soundiiz.